In [ ]:
# =================================================================
# Sprint 52 Task A — Prepare a Mandarin Dataset for Human Review
# =================================================================
# Continues Sprint 51 Task 1 (finetune_enzh_train.jsonl + split pipeline).
#
# Goal: produce a fresh EN<->CMN sample set, independently sourced from
# the Sprint 51 training data, spanning easy/medium/hard quality tiers,
# with full provenance — packaged with candidate model output so human
# reviewers can score translation quality using the project's 6-dimension
# rubric (Semantic Accuracy, Fluency, Tone/Register, Emotional Consistency,
# Cultural Appropriateness, Dialect Correctness).
#
# Run this AFTER Sprint 51's fine-tuning notebook has produced a checkpoint
# and candidate_manifest.json (used here to load the model and to exclude
# any pairs already seen during training).
# =================================================================

!pip install -q -U datasets huggingface_hub
!pip install -q unsloth tqdm sacremoses

import os
import json
import random
import hashlib
import logging
import unicodedata
import torch

from pathlib import Path
from tqdm.auto import tqdm
from datasets import load_dataset
from collections import defaultdict
from unsloth import FastLanguageModel
from datetime import datetime, timezone
from kaggle_secrets import UserSecretsClient

print("✓ Imports ready")

In [ ]:
# Cells for defining: PATHS ; CONFIG ; UTILS
# ========================================= PATHS ==========================================

SPRINT51_TRAIN_MANIFEST = Path("/kaggle/input/sprint51-outputs/manifests/split_manifest.json")

SPRINT51_TRAIN_FILES = [
    Path("/kaggle/input/notebooks/abdighaz/sprint51-task1-mandarin-data-split/splits/finetune.jsonl"),
    Path("/kaggle/input/notebooks/abdighaz/sprint51-task1-mandarin-data-split/splits/judge_calibration.jsonl"),
    Path("/kaggle/input/notebooks/abdighaz/sprint51-task1-mandarin-data-split/splits/prompt_validation.jsonl"),
    Path("/kaggle/input/notebooks/abdighaz/sprint51-task1-mandarin-data-split/splits/sealed_final_test.jsonl"),
]

CHECKPOINT_DIR = Path(
    "/kaggle/input/notebooks/abdighaz/sprint51-task1-full-finetuning-pipeline/task1_en2zh_outputs/final"
)

OUTPUT_DIR = Path("/kaggle/working/review_package")
LOGS = Path("/kaggle/working/logs")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
LOGS.mkdir(parents=True, exist_ok=True)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    handlers=[
        logging.FileHandler(LOGS / "taskA_pipeline.log", encoding="utf-8"),
        logging.StreamHandler(),
    ],
)

# Fetch token from Kaggle Secrets and set it in the environment
user_secrets = UserSecretsClient()
os.environ["HF_TOKEN"] = user_secrets.get_secret("kaggle-glm-eval")

# ======================================== CONFIG =========================================

DIRECTIONS = ["en2cmn"]  # ["en2cmn", "cmn2en"]

TARGET_PER_DIRECTION_PER_TIER = {
    "clean": 40,       # FLORES-200 devtest — formal, professionally translated
    "domain": 40,      # WMT19 newstest en-zh — news domain, longer sentences
    "colloquial": 40,  # OpenSubtitles-en-zh-cn-20m Movie / TV dialogue
}

TARGET_PER_DIRECTION = sum(TARGET_PER_DIRECTION_PER_TIER.values())  # 120

SOURCES = {
    "clean": {
        "dataset_name": "facebook/flores",
        "config": "eng_Latn-zho_Hans",
        "license": "CC-BY-SA 4.0",
    },
    "domain": {
        "dataset_name": "wmt/wmt19",
        "config": "zh-en",
        "license": "Unknown",
    },
    "colloquial": {
        "dataset_name": "FradSer/OpenSubtitles-en-zh-cn-20m",
        "config": "default",
        "license": "MIT",
    },
}

LENGTH_THRESHOLDS = {
    "clean": {"min_chars": 15, "max_chars": 300},
    "domain": {"min_chars": 15, "max_chars": 300},
    "colloquial": {"min_chars": 2, "max_chars": 300},
}

RANDOM_SEED = 52
random.seed(RANDOM_SEED)

RUN_DATE = datetime.now(timezone.utc).strftime("%Y-%m-%d")

# ======================================== UTILS =========================================

def sha256_text(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


def normalize_text(text):
    if text is None:
        return None

    text = unicodedata.normalize("NFKC", str(text))
    text = text.replace("\u00a0", " ").strip()

    return " ".join(text.split())


def contains_cjk(text: str) -> bool:
    return any("\u4e00" <= ch <= "\u9fff" for ch in text)


def passes_length(text: str, tier: str = "clean") -> bool:
    thresholds = LENGTH_THRESHOLDS[tier]
    return thresholds["min_chars"] <= len(text) <= thresholds["max_chars"]


print("✓ defining: PATHS ; CONFIG ; UTILS")
print(f"✓ Directions: {DIRECTIONS}")
print(f"✓ Target rows per direction: {TARGET_PER_DIRECTION}")
print(f"✓ Target per tier: {TARGET_PER_DIRECTION_PER_TIER}")
print(f"✓ Length thresholds: {LENGTH_THRESHOLDS}")
print(f"✓ Random seed: {RANDOM_SEED}")

In [ ]:
# ====================== STEP 1: build exclusion set from Sprint 51 ======================

def build_exclusion_set():
    """Hash every source/target sentence already used in Sprint 51 training
    and eval splits, so Task A can guarantee independence."""

    excluded = set()
    n_loaded = 0

    for path in SPRINT51_TRAIN_FILES:
        if not path.exists():
            logging.warning(f"Sprint 51 split not found, skipping: {path}")
            continue

        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                try:
                    rec = json.loads(line)
                except json.JSONDecodeError:
                    continue

                src = normalize_text(rec.get("source_text"))
                tgt = normalize_text(rec.get("target_text"))

                if src:
                    excluded.add(sha256_text(src))

                if tgt:
                    excluded.add(sha256_text(tgt))

                n_loaded += 1

    logging.info(
        f"Exclusion set built from {n_loaded:,} Sprint 51 records "
        f"({len(excluded):,} unique hashes)."
    )

    return excluded


print("✓ STEP 1: build exclusion set from Sprint 51")

In [ ]:
# ====================== STEP 2: load each tier's independent source ======================

def load_clean_tier():
    """FLORES-200 devtest, English <-> Simplified Chinese."""

    ds = load_dataset(
        SOURCES["clean"]["dataset_name"],
        SOURCES["clean"]["config"],
        split="devtest",
        streaming=True,
    )

    pairs = []

    for i, row in enumerate(ds):
        en = normalize_text(row.get("sentence_eng_Latn"))
        zh = normalize_text(row.get("sentence_zho_Hans"))

        if en and zh:
            pairs.append({
                "en": en,
                "zh": zh,
                "source_example_id": f"flores200_devtest_{i}",
            })

    return pairs

def load_domain_tier(target_size=1012):
    """WMT19 en-zh newstest — news domain."""

    ds = load_dataset(
        SOURCES["domain"]["dataset_name"],
        SOURCES["domain"]["config"],
        split="validation",
        streaming=True,
    )

    pairs = []

    for i, row in enumerate(ds):
        if len(pairs) >= target_size:
            break

        translation = row.get("translation", {})

        en = normalize_text(translation.get("en"))
        zh = normalize_text(translation.get("zh"))

        if en and zh:
            pairs.append({
                "en": en,
                "zh": zh,
                "source_example_id": f"wmt19_newstest_{i}",
            })

    return pairs


def load_colloquial_tier(target_size=200):
    """OpenSubtitles English–Simplified Chinese subtitles.
    """

    logging.info(
        "[colloquial] Opening OpenSubtitles streaming dataset..."
    )

    ds = load_dataset(
        SOURCES["colloquial"]["dataset_name"],
        SOURCES["colloquial"]["config"],
        split="train",
        streaming=True,
        token=os.environ.get("HF_TOKEN"),
    )

    pairs = []

    min_chars = LENGTH_THRESHOLDS["colloquial"]["min_chars"]
    max_chars = LENGTH_THRESHOLDS["colloquial"]["max_chars"]

    logging.info(
        f"[colloquial] Looking for {target_size} eligible pairs..."
    )

    for i, row in enumerate(ds):

        # OpenSubtitles dataset uses `source` and `target`
        en = normalize_text(row.get("source"))
        zh = normalize_text(row.get("target"))

        if not en or not zh:
            continue

        # Length filtering
        if not (
            min_chars <= len(en) <= max_chars
            and min_chars <= len(zh) <= max_chars
        ):
            continue

        # Require Chinese characters in target
        if not contains_cjk(zh):
            continue

        pairs.append({
            "en": en,
            "zh": zh,
            "source_example_id": f"opensubtitles_en_zh_{i}",
        })

        if len(pairs) % 10 == 0:
            logging.info(
                f"[colloquial] collected "
                f"{len(pairs)}/{target_size} eligible pairs"
            )

        if len(pairs) >= target_size:
            break

    logging.info(
        f"[colloquial] collected {len(pairs)} eligible pairs "
        f"after scanning {i + 1:,} records."
    )

    return pairs

TIER_LOADERS = {
    "clean": load_clean_tier,
    "domain": load_domain_tier,
    "colloquial": load_colloquial_tier,
}

print("✓ STEP 2: load each tier's independent source")

In [ ]:

# ====================== STEP 3: filter, dedupe, sample ======================

def filter_and_sample(pairs, tier, excluded_hashes, n_needed):
    kept = []

    for p in pairs:
        min_chars = LENGTH_THRESHOLDS[tier]["min_chars"]
        max_chars = LENGTH_THRESHOLDS[tier]["max_chars"]

        if not (
            min_chars <= len(p["en"]) <= max_chars
            and min_chars <= len(p["zh"]) <= max_chars
        ):
            continue

        if not contains_cjk(p["zh"]):
            continue

        if (
            sha256_text(p["en"]) in excluded_hashes
            or sha256_text(p["zh"]) in excluded_hashes
        ):
            continue  # overlaps Sprint 51 training/eval data — reject

        kept.append(p)

    random.shuffle(kept)
    selected = kept[:n_needed]

    logging.info(
        f"[{tier}] eligible after filtering/dedup: "
        f"{len(kept):,} | selected: {len(selected):,}"
    )

    if len(selected) < n_needed:
        logging.warning(
            f"[{tier}] only found {len(selected)} of {n_needed} requested — "
            f"widen LENGTH_THRESHOLDS or pull a larger split."
        )

    return selected


print("✓ STEP 3: filter, dedupe, sample")
print("✓ Tier-specific length filtering enabled")
print("✓ Sprint 51 overlap exclusion enabled")
print("✓ CJK validation enabled")

In [ ]:
# ====================== STEP 4: generate candidate translations ======================

CANDIDATE_SYSTEM_PROMPT = (
    "You are an expert translator for English to Mandarin Chinese (普通話). "
    "Rules you must follow:\n"
    "1. Output ONLY the Mandarin translation in Simplified Chinese characters.\n"
    "2. Use standard Mainland China Mandarin vocabulary and grammar.\n"
    "3. Do NOT use Cantonese vocabulary or particles.\n"
    "4. Do NOT romanize and do not use Pinyin.\n"
    "5. Do NOT explain, repeat the source, or add commentary.\n"
    "6. If a proper noun has no Mandarin equivalent, keep the original English term."
)


def load_finetuned_model():
    print("[MODEL] Loading Sprint 51 fine-tuned checkpoint...")
    print(f"[MODEL] Checkpoint: {CHECKPOINT_DIR}")

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=str(CHECKPOINT_DIR),
        max_seq_length=1024,
        dtype=None,
        load_in_4bit=True,
        device_map={"": 0},
    )

    FastLanguageModel.for_inference(model)

    model.generation_config.max_length = None

    print("[MODEL] ✓ Checkpoint loaded successfully")
    print(f"[MODEL] Device: {model.get_input_embeddings().weight.device}")

    return model, tokenizer


def translate(model, tokenizer, system_prompt, src_text, max_new_tokens=256):
    messages = [
        {
            "role": "system",
            "content": system_prompt,
        },
        {
            "role": "user",
            "content": f"Translate to Mandarin:\n{src_text}",
        },
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    input_device = model.get_input_embeddings().weight.device

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512,
        padding=False,
    )

    inputs = {
        key: value.to(input_device)
        for key, value in inputs.items()
    }

    input_length = inputs["input_ids"].shape[1]

    with torch.inference_mode():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            use_cache=True,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated_tokens = out[0][input_length:]

    candidate = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True,
    )

    return normalize_text(candidate)

print("✓ STEP 4: generate new candidate translations")
print(f"✓ Checkpoint: {CHECKPOINT_DIR}")
print("✓ Single-device inference configured: cuda:0")

In [ ]:
# ====================== STEP 5: assemble review rows ======================

RUBRIC_DIMENSIONS = [
    "Semantic_Accuracy",
    "Fluency",
    "Tone_Register",
    "Emotional_Consistency",
    "Cultural_Appropriateness",
    "Dialect_Correctness",
]

def build_review_rows(sampled_by_tier, model, tokenizer):
    rows = []
    counter = 0

    for tier, pairs in sampled_by_tier.items():
        source_info = SOURCES[tier]

        for p in pairs:
            for direction in DIRECTIONS:
                counter += 1

                if direction == "en2cmn":
                    source_text = p["en"]
                    reference_text = p["zh"]
                else:
                    source_text = p["zh"]
                    reference_text = p["en"]

                candidate_text = translate(
                    model,
                    tokenizer,
                    CANDIDATE_SYSTEM_PROMPT,
                    source_text,
                )

                print(f"\n[{counter}]")
                print("EN:", source_text)
                print("CMN:", repr(candidate_text))

                row = {
                    "id": f"s52_taskA_{counter:05d}",
                    "direction": direction,
                    "quality_tier": tier,
                    "source_text": source_text,
                    "reference_text": reference_text,
                    "candidate_text": candidate_text,
                    "source_dataset": source_info["dataset_name"],
                    "source_example_id": p["source_example_id"],
                    "license": source_info["license"],
                    "retrieval_date": RUN_DATE,
                    "dedup_check": "passed",
                }

                for dim in RUBRIC_DIMENSIONS:
                    row[dim] = ""

                row["Overall_Comments"] = ""

                rows.append(row)

    return rows


print("✓ STEP 5: assemble review rows")
print(f"✓ Rubric dimensions: {len(RUBRIC_DIMENSIONS)}")

In [ ]:
# ====================== MAIN CELL ======================

print("=" * 70)
print("SPRINT 52 TASK A — PIPELINE START")
print("=" * 70)

# -----------------------------------------------------------------
# STEP 1 — Build exclusion set
# -----------------------------------------------------------------

print("\n[STEP 1/5] Building Sprint 51 exclusion set...")

excluded_hashes = build_exclusion_set()

print(
    f"✓ STEP 1 COMPLETE | "
    f"{len(excluded_hashes):,} unique hashes loaded"
)

# -----------------------------------------------------------------
# STEP 2 + STEP 3 — Load, filter and sample each tier
# -----------------------------------------------------------------

print("\n[STEP 2/5] Loading and filtering source datasets...")

sampled_by_tier = {}

for tier, n_needed in TARGET_PER_DIRECTION_PER_TIER.items():

    print(
        f"\n  → Loading tier: {tier} | "
        f"target: {n_needed}"
    )

    logging.info(
        f"Loading tier '{tier}' from "
        f"{SOURCES[tier]['dataset_name']}..."
    )

    raw_pairs = TIER_LOADERS[tier]()

    print(
        f"  ✓ Raw candidate pairs loaded: "
        f"{len(raw_pairs):,}"
    )

    sampled_by_tier[tier] = filter_and_sample(
        raw_pairs,
        tier,
        excluded_hashes,
        n_needed,
    )

    print(
        f"  ✓ Selected {len(sampled_by_tier[tier])}/"
        f"{n_needed} pairs for {tier}"
    )

print("\n✓ STEP 2/3 COMPLETE — all tiers processed")

# -----------------------------------------------------------------
# Sampling summary
# -----------------------------------------------------------------

print("\n[SUMMARY] Sampled pairs by tier:")

for tier, pairs in sampled_by_tier.items():
    print(f"  • {tier:12s}: {len(pairs):3d}")

total_pairs = sum(
    len(pairs) for pairs in sampled_by_tier.values()
)

print(f"  • {'TOTAL':12s}: {total_pairs:3d}")

# -----------------------------------------------------------------
# STEP 4 — Load model
# -----------------------------------------------------------------

print("\n[STEP 4/5] Loading fine-tuned checkpoint...")

logging.info("Loading fine-tuned checkpoint for candidate generation...")

model, tokenizer = load_finetuned_model()

print("✓ STEP 4 COMPLETE — model ready for inference")

# -----------------------------------------------------------------
# STEP 5 — Generate candidate translations
# -----------------------------------------------------------------

print("\n[STEP 5/5] Generating candidate translations...")

rows = build_review_rows(
    sampled_by_tier,
    model,
    tokenizer,
)

print(
    f"✓ Candidate generation complete | "
    f"{len(rows):,} review rows created"
)

# -----------------------------------------------------------------
# Safety check
# -----------------------------------------------------------------

if not rows:
    raise RuntimeError(
        "No review rows were generated. "
        "Check dataset loading/filtering before saving outputs."
    )

expected_rows = (
    sum(TARGET_PER_DIRECTION_PER_TIER.values())
    * len(DIRECTIONS)
)

print(
    f"✓ Expected maximum rows: {expected_rows:,} | "
    f"Actual rows: {len(rows):,}"
)

# -----------------------------------------------------------------
# Save CSV
# -----------------------------------------------------------------

print("\n[SAVE 1/3] Writing CSV review file...")

import csv

csv_path = OUTPUT_DIR / "mandarin_human_review_set.csv"
fieldnames = list(rows[0].keys())

with open(
    csv_path,
    "w",
    newline="",
    encoding="utf-8-sig",
) as f:
    writer = csv.DictWriter(
        f,
        fieldnames=fieldnames,
    )
    writer.writeheader()
    writer.writerows(rows)

print(f"✓ CSV saved: {csv_path}")

# -----------------------------------------------------------------
# Save JSONL
# -----------------------------------------------------------------

print("\n[SAVE 2/3] Writing JSONL review file...")

jsonl_path = OUTPUT_DIR / "mandarin_human_review_set.jsonl"

with open(
    jsonl_path,
    "w",
    encoding="utf-8",
) as f:
    for row in rows:
        f.write(
            json.dumps(
                row,
                ensure_ascii=False,
            )
            + "\n"
        )

print(f"✓ JSONL saved: {jsonl_path}")

# -----------------------------------------------------------------
# Manifest
# -----------------------------------------------------------------

print("\n[SAVE 3/3] Writing review-set manifest...")

counts_by_direction_tier = defaultdict(int)

for row in rows:
    counts_by_direction_tier[
        f"{row['direction']}_{row['quality_tier']}"
    ] += 1

manifest = {
    "sprint": "Sprint 52 Task A",
    "generated": RUN_DATE,
    "target_per_direction": TARGET_PER_DIRECTION,
    "target_per_direction_per_tier": TARGET_PER_DIRECTION_PER_TIER,
    "directions": DIRECTIONS,
    "sources": SOURCES,
    "random_seed": RANDOM_SEED,
    "length_thresholds": LENGTH_THRESHOLDS,
    "excluded_hash_count": len(excluded_hashes),
    "total_rows": len(rows),
    "counts_by_direction_tier": dict(counts_by_direction_tier),
    "rubric_dimensions": RUBRIC_DIMENSIONS,
    "file_hashes": {
        "csv": sha256_file(csv_path),
        "jsonl": sha256_file(jsonl_path),
    },
}

manifest_path = OUTPUT_DIR / "review_set_manifest.json"

with open(
    manifest_path,
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        manifest,
        f,
        indent=2,
        ensure_ascii=False,
    )

print(f"✓ Manifest saved: {manifest_path}")

# -----------------------------------------------------------------
# Final summary
# -----------------------------------------------------------------

logging.info(
    f"Done. {len(rows)} review rows written to {csv_path}"
)
logging.info(f"Manifest: {manifest_path}")

print("\n" + "=" * 70)
print("SPRINT 52 TASK A — PIPELINE COMPLETE")
print("=" * 70)

print(f"✓ Total review rows: {len(rows):,}")
print(f"✓ CSV:   {csv_path}")
print(f"✓ JSONL: {jsonl_path}")
print(f"✓ Manifest: {manifest_path}")

print("\nCounts by direction/tier:")

for key, count in sorted(counts_by_direction_tier.items()):
    print(f"  • {key:20s}: {count:3d}")

print("=" * 70)